# Data Slayer 4.0: Exam Cheating Detection
## Baseline Pipeline: LeVJEPA 1.5 Large + LightGBM (GroupKFold)

This notebook implements an end-to-end baseline for the **Data Slayer 4.0** competition:
1. **Backbone**: `galilai-group/LeVJEPA-1.5-Large` (Self-Supervised Video ViT-Large, frozen feature extractor)
2. **Frame Sampling**: 16 uniform frames per video (`.mov` for train, `.jpg` sequences for test)
3. **Cross-Validation**: 5-Fold `GroupKFold` grouped by **Subject ID** (strict zero-leakage)
4. **Classifier**: `LightGBM` (fast, immune to high-dim overfitting, easy to blend)
5. **Post-Processing**: Global Threshold Search to maximize **F1-Macro Score**

> **Note:** Make sure **GPU accelerator (T4 x1 or P100)** and **Internet ON** are enabled in your Kaggle Notebook settings to download the model weights.

In [ ]:
# Install dependencies (if not pre-installed)
!pip install -q transformers timm lightgbm opencv-python scikit-learn decord

In [ ]:
import os
import glob
import re
import cv2
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import AutoModel

import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report

# Set seeds for reproducibility
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
# Global Configurations
class Config:
    # Verified competition dataset path
    DATA_DIR = "/kaggle/input/competitions/data-slayer-4" if os.path.exists("/kaggle/input/competitions/data-slayer-4") else "/kaggle/input/data-slayer-4"
    MODEL_NAME = "galilai-group/LeVJEPA-1.5-Large"
    NUM_FRAMES = 16          # LeVJEPA standard temporal window
    IMG_SIZE = 224           # ViT-Large input spatial resolution
    BATCH_SIZE = 8           # Fits comfortably in 16GB T4 GPU
    NUM_WORKERS = 2
    N_FOLDS = 5
    OUTPUT_DIR = "/kaggle/working"

cfg = Config()
print(f"Dataset Directory set to: {cfg.DATA_DIR}")

### 1. Build Metadata & File Manifest
- Scan `/kaggle/input/data-slayer-4/train/` to extract `video_path`, `label`, and `subject`.
- Scan `/kaggle/input/data-slayer-4/test/` to list all sequence folders.

In [ ]:
# 1. Collect Train Data (Robust case-insensitive crawler)
train_records = []
train_root = os.path.join(cfg.DATA_DIR, "train")
if not os.path.exists(train_root):
    # Fallback search for 'train' folder
    for item in os.listdir(cfg.DATA_DIR):
        if item.lower() == "train":
            train_root = os.path.join(cfg.DATA_DIR, item)
            break

print(f"Scanning train directory: {train_root}")
if train_root and os.path.exists(train_root):
    for root, dirs, files in os.walk(train_root):
        for file in files:
            if file.lower().endswith(('.mov', '.mp4', '.avi', '.mkv')):
                v_path = os.path.join(root, file)
                path_lower = v_path.lower()
                
                # Determine label from directory path
                if "cheat" in path_lower:
                    label_id = 1
                    label_name = "cheat"
                elif "normal" in path_lower:
                    label_id = 0
                    label_name = "normal"
                else:
                    continue
                
                # Extract subject (e.g. subject_1)
                subject = "unknown"
                for part in os.path.normpath(v_path).split(os.sep):
                    if "subject" in part.lower():
                        subject = part
                        break
                        
                train_records.append({
                    "video_path": v_path,
                    "label": label_id,
                    "label_name": label_name,
                    "subject": subject,
                    "filename": file
                })

df_train = pd.DataFrame(train_records)
print(f"Total Train Videos found: {len(df_train)}")
if len(df_train) > 0:
    print(f"Unique Subjects: {df_train['subject'].nunique()}")
    print(df_train['label_name'].value_counts())
    display(df_train.head())
else:
    print("Error: No videos found! Please check train_root contents:", os.listdir(train_root))

In [ ]:
# 2. Collect Test Data
sample_sub_path = os.path.join(cfg.DATA_DIR, "sample_submission.csv")
df_sub = pd.read_csv(sample_sub_path)
print(f"Sample submission shape: {df_sub.shape}")
display(df_sub.head())

# Check test sequences
test_dir = os.path.join(cfg.DATA_DIR, "test")
# Ensure test sequences match the order expected in sample_submission
id_col = df_sub.columns[0]
test_seq_paths = []
for seq_id in df_sub[id_col]:
    seq_folder = os.path.join(test_dir, str(seq_id))
    test_seq_paths.append(seq_folder)

df_test = pd.DataFrame({
    "seq_id": df_sub[id_col],
    "folder_path": test_seq_paths
})
print(f"Total Test Sequences: {len(df_test)}")
display(df_test.head())

### 2. Video & Frame Loading Utilities
- **Train loader:** Extracts 16 uniformly sampled frames from `.mov` files.
- **Test loader:** Loads 16 uniformly sampled frames from `01.jpg` to `30.jpg` in each sequence folder.
- Output format for LeVJEPA: `(3, 16, 224, 224)` tensor with ImageNet normalization.

In [ ]:
# Optimized ImageNet normalization constants
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1, 1)

try:
    from decord import VideoReader, cpu
    HAS_DECORD = True
except ImportError:
    HAS_DECORD = False

def load_video_frames(video_path, num_frames=16):
    """
    High-speed video frame extraction:
    Uses Decord (C++ multithreaded) to jump directly to target frames without decoding intermediate frames.
    Falls back to direct OpenCV seek if needed.
    """
    if HAS_DECORD:
        try:
            vr = VideoReader(video_path, ctx=cpu(0), num_threads=1)
            total_frames = len(vr)
            if total_frames > 0:
                indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
                # Instantly extract only the 16 target frames
                raw_frames = vr.get_batch(indices).asnumpy()  # (16, H, W, 3)
                resized = [cv2.resize(f, (cfg.IMG_SIZE, cfg.IMG_SIZE)) for f in raw_frames]
                arr = np.stack(resized)  # (16, 224, 224, 3)
                tensor = torch.from_numpy(arr).permute(3, 0, 1, 2).float() / 255.0  # (3, 16, 224, 224)
                return (tensor - IMAGENET_MEAN) / IMAGENET_STD
        except Exception:
            pass

    # Fallback: OpenCV Direct Frame Seek (skips reading all intervening frames)
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, max(total_frames - 1, 1), num_frames, dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (cfg.IMG_SIZE, cfg.IMG_SIZE))
        else:
            frame = np.zeros((cfg.IMG_SIZE, cfg.IMG_SIZE, 3), dtype=np.uint8)
        frames.append(frame)
    cap.release()
    
    arr = np.stack(frames)
    tensor = torch.from_numpy(arr).permute(3, 0, 1, 2).float() / 255.0
    return (tensor - IMAGENET_MEAN) / IMAGENET_STD

def load_sequence_frames(folder_path, num_frames=16):
    """
    Fast loader for test sequence JPG frames.
    """
    jpg_files = sorted(glob.glob(os.path.join(folder_path, "*.jpg")) + glob.glob(os.path.join(folder_path, "*.png")))
    total_frames = len(jpg_files)
    if total_frames == 0:
        return torch.zeros((3, num_frames, cfg.IMG_SIZE, cfg.IMG_SIZE))
        
    indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    frames = []
    for idx in indices:
        img = cv2.imread(jpg_files[idx])
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (cfg.IMG_SIZE, cfg.IMG_SIZE))
        else:
            img = np.zeros((cfg.IMG_SIZE, cfg.IMG_SIZE, 3), dtype=np.uint8)
        frames.append(img)
        
    arr = np.stack(frames)
    tensor = torch.from_numpy(arr).permute(3, 0, 1, 2).float() / 255.0
    return (tensor - IMAGENET_MEAN) / IMAGENET_STD

In [ ]:
# PyTorch Dataset classes
class CheatingTrainDataset(Dataset):
    def __init__(self, df, num_frames=16):
        self.df = df
        self.num_frames = num_frames
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tensor = load_video_frames(row["video_path"], self.num_frames)
        label = row["label"]
        return tensor, label, row["subject"]

class CheatingTestDataset(Dataset):
    def __init__(self, df, num_frames=16):
        self.df = df
        self.num_frames = num_frames
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tensor = load_sequence_frames(row["folder_path"], self.num_frames)
        return tensor, row["seq_id"]

### 3. Feature Extraction with LeVJEPA-1.5-Large
- We load the pretrained LeVJEPA model in `float16`.
- Backbone weights are completely frozen (`torch.no_grad()`).
- Features are cached to `.npy` so that LightGBM can be re-trained and tuned in seconds without re-running the neural network.

In [ ]:
print(f"Loading {cfg.MODEL_NAME}...")
model = AutoModel.from_pretrained(cfg.MODEL_NAME, trust_remote_code=True)
model = model.half().to(DEVICE).eval()
print("Model loaded successfully!")

In [ ]:
# Extract features for Training set
train_dataset = CheatingTrainDataset(df_train, num_frames=cfg.NUM_FRAMES)
train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=cfg.NUM_WORKERS)

train_features = []
train_labels = []
train_subjects = []

with torch.no_grad():
    for tensors, labels, subjects in tqdm(train_loader, desc="Extracting Train Features"):
        tensors = tensors.half().to(DEVICE)
        outputs = model(pixel_values=tensors)
        
        # Option: pooler_output (1024-dim)
        if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
            feats = outputs.pooler_output.cpu().numpy()
        else:
            # Fallback: global mean of patch tokens
            feats = outputs.last_hidden_state[:, 1:, :].mean(dim=1).cpu().numpy()
            
        train_features.append(feats)
        train_labels.extend(labels.numpy())
        train_subjects.extend(subjects)

X_train = np.vstack(train_features)
y_train = np.array(train_labels)
subjects_train = np.array(train_subjects)

print(f"Extracted X_train shape: {X_train.shape}")
np.save(os.path.join(cfg.OUTPUT_DIR, "X_train.npy"), X_train)
np.save(os.path.join(cfg.OUTPUT_DIR, "y_train.npy"), y_train)
np.save(os.path.join(cfg.OUTPUT_DIR, "subjects_train.npy"), subjects_train)

In [ ]:
# Extract features for Test set
test_dataset = CheatingTestDataset(df_test, num_frames=cfg.NUM_FRAMES)
test_loader = DataLoader(test_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=cfg.NUM_WORKERS)

test_features = []
test_seq_ids = []

with torch.no_grad():
    for tensors, seq_ids in tqdm(test_loader, desc="Extracting Test Features"):
        tensors = tensors.half().to(DEVICE)
        outputs = model(pixel_values=tensors)
        
        if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
            feats = outputs.pooler_output.cpu().numpy()
        else:
            feats = outputs.last_hidden_state[:, 1:, :].mean(dim=1).cpu().numpy()
            
        test_features.append(feats)
        test_seq_ids.extend(seq_ids)

X_test = np.vstack(test_features)
print(f"Extracted X_test shape: {X_test.shape}")
np.save(os.path.join(cfg.OUTPUT_DIR, "X_test.npy"), X_test)

# Free GPU memory for downstream modeling
del model
torch.cuda.empty_cache()

### 4. GroupKFold Cross-Validation with LightGBM
- We split strictly by `subjects_train` so validation subjects are **never** seen in training.
- LightGBM hyperparameters are chosen to avoid overfitting in the small sample regime ($N=416, D=1024$).

In [ ]:
gkf = GroupKFold(n_splits=cfg.N_FOLDS)

oof_preds = np.zeros(len(y_train))
test_preds = np.zeros(len(X_test))
models = []

lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'num_leaves': 15,
    'max_depth': 4,
    'min_data_in_leaf': 10,
    'feature_fraction': 0.5,    # Subsample 50% features per tree to prevent overfitting
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'lambda_l1': 0.1,
    'lambda_l2': 1.0,
    'verbose': -1,
    'random_state': 42
}

print(f"Starting {cfg.N_FOLDS}-Fold GroupKFold Training...\n")
for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups=subjects_train)):
    X_tr, y_tr = X_train[train_idx], y_train[train_idx]
    X_va, y_val = X_train[val_idx], y_train[val_idx]
    
    train_data = lgb.Dataset(X_tr, label=y_tr)
    val_data = lgb.Dataset(X_va, label=y_val, reference=train_data)
    
    bst = lgb.train(
        lgb_params,
        train_data,
        num_boost_round=500,
        valid_sets=[train_data, val_data],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
    )
    
    val_pred = bst.predict(X_va, num_iteration=bst.best_iteration)
    oof_preds[val_idx] = val_pred
    
    test_preds += bst.predict(X_test, num_iteration=bst.best_iteration) / cfg.N_FOLDS
    models.append(bst)
    
    fold_f1 = f1_score(y_val, (val_pred >= 0.5).astype(int), average='macro')
    print(f"Fold {fold + 1} | Best Iter: {bst.best_iteration} | Val F1-Macro (th=0.5): {fold_f1:.4f}")

### 5. Threshold Optimization for F1-Macro
In competitive evaluation, the default 0.5 probability threshold is rarely optimal for F1-Macro. We search across $[0.1, 0.9]$ to find the exact threshold that maximizes out-of-fold F1-Macro.

In [ ]:
best_th = 0.5
best_f1 = 0.0

thresholds = np.linspace(0.1, 0.9, 81)
for th in thresholds:
    score = f1_score(y_train, (oof_preds >= th).astype(int), average='macro')
    if score > best_f1:
        best_f1 = score
        best_th = th

print("=" * 40)
print(f"Default Threshold (0.50) OOF F1-Macro: {f1_score(y_train, (oof_preds >= 0.5).astype(int), average='macro'):.4f}")
print(f"Optimized Threshold ({best_th:.2f}) OOF F1-Macro: {best_f1:.4f}")
print("=" * 40)

# Detailed Classification Report
oof_final = (oof_preds >= best_th).astype(int)
print("\nOut-Of-Fold Classification Report:")
print(classification_report(y_train, oof_final, target_names=["Normal (0)", "Cheat (1)"]))

### 6. Generate Submission File
- Format predictions to match `sample_submission.csv` columns and row order.

In [ ]:
# Apply optimal threshold
final_test_labels = (test_preds >= best_th).astype(int)

# Create submission DataFrame
df_submission = df_sub.copy()
target_col = df_sub.columns[1] # Target column name (usually 'label' or 'target')
df_submission[target_col] = final_test_labels

sub_out_path = os.path.join(cfg.OUTPUT_DIR, "submission.csv")
df_submission.to_csv(sub_out_path, index=False)
print(f"Saved submission to {sub_out_path}")

print("\nPrediction Class Distribution in Test:")
print(df_submission[target_col].value_counts())
display(df_submission.head(10))